# PDF2TABLE

## 📦 Setup and Imports

In this block, we:
- Install all required packages including LangChain, ChromaDB, Streamlit, and PDF tools
- Import modules for document loading, text splitting, vector embedding, prompt creation, and structured data modeling
- Load our OpenAI API key securely from a `.env` file using `dotenv`

In [1]:
# Install required packages
!pip3 install --upgrade --quiet langchain langchain-community langchain-openai chromadb 
!pip3 install --upgrade --quiet pypdf pandas streamlit python-dotenv

# Import LangChain modules
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# Other modules and packages
import os
import tempfile
import streamlit as st  
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 🤖 Define and Test the Language Model (LLM)

In this block, we initialize the `ChatOpenAI` model (e.g., GPT-4o Mini) using our API key and make a sample request to verify everything is working. This confirms that the OpenAI connection and model access are set up correctly.

In [3]:
# Define our LLM
llm = ChatOpenAI(model="gpt-4o-mini")
response = llm.invoke("Tell me a joke about global warming")
print(response)

content="Why did the climate scientist break up with their partner?\n\nBecause they just couldn't handle the heat!" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 14, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-C6oq11wXNDEhqjwg2FFoxRyJj1xeG', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--92a282b0-d98a-43d1-a4b5-a71ad80e7227-0' usage_metadata={'input_tokens': 14, 'output_tokens': 19, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## 📄 Load and Chunk the PDF Document

In this step, we:
- Load the PDF using `PyPDFLoader` from LangChain
- Split it into smaller overlapping chunks using `RecursiveCharacterTextSplitter`  
  This ensures that the language model can process each chunk efficiently and with contextual continuity.

In [4]:
### 📄 Load PDF document
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

pdf_path = "data/Oppenheimer-2006-Applied_Cognitive_Psychology.pdf"
assert os.path.exists(pdf_path), f"❌ PDF file not found at {pdf_path}"

# Load PDF
loader = PyPDFLoader(pdf_path)
pages = loader.load()
print(f"✅ Loaded {len(pages)} pages from PDF")
print(pages[0].page_content[:500])  # Preview first 500 characters of the first page

### ✂️ Split documents into chunks
if pages:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", " "]
    )
    chunks = text_splitter.split_documents(pages)
    print(f"✅ Split into {len(chunks)} text chunks")
else:
    chunks = []
    print("⚠️ No pages found to split.")

Ignoring wrong pointing object 18 0 (offset 0)


✅ Loaded 3 pages from PDF
APPLIED COGNITIVE PSYCHOLOGY
Appl. Cognit. Psychol. 20: 139–156 (2006)
Published online 31 October 2005 in Wiley InterScience
(www.interscience.wiley.com) DOI: 10.1002/acp.1178
Consequences of Erudite Vernacular Utilized Irrespective
of Necessity: Problems with Using Long Words Needlessly
DANIEL M. OPPENHEIMER*
Princeton University, USA
SUMMARY
Most texts on writing style encourage authors to avoid overly-complex words. However, a majority
of undergraduates admit to deliberately increasing the c
✅ Split into 9 text chunks


## 🧠 Embed Text Chunks & Create Vector Database

In this block, we:
- Define the OpenAI embedding model (`text-embedding-3-small`)
- Evaluate how semantically close two strings are using embedding distance
- Assign unique IDs to each document chunk using UUID v5
- Store all unique embeddings in a persistent Chroma vector database for efficient semantic search

In [5]:
### 🔢 Define Embedding Function
from langchain_openai import OpenAIEmbeddings

def get_embedding_function():
    return OpenAIEmbeddings(
        model="text-embedding-3-small",  # Updated to newer, more efficient model
        openai_api_key=OPENAI_API_KEY
    )

embedding_function = get_embedding_function()

# Test embedding
test_vector = embedding_function.embed_query("cat")
print(f"✅ Embedding vector length: {len(test_vector)}")

### 📏 Evaluate Embedding Distance
from langchain.evaluation import load_evaluator

evaluator = load_evaluator(
    evaluator="embedding_distance", 
    embeddings=embedding_function
)

# More relevant examples for research papers
score1 = evaluator.evaluate_strings(prediction="machine learning", reference="artificial intelligence")
score2 = evaluator.evaluate_strings(prediction="data mining", reference="artificial intelligence")

print(f"🔍 Distance (machine learning vs AI): {score1['score']:.4f}")
print(f"🔍 Distance (data mining vs AI): {score2['score']:.4f}")

### 🧠 Create Vectorstore from Unique Chunks
import uuid
from langchain_community.vectorstores import Chroma  # Fixed import

def create_vectorstore(chunks, embedding_function, vectorstore_path):
    # Create unique UUIDs based on chunk content
    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, doc.page_content)) for doc in chunks]
    
    unique_ids = set()
    unique_chunks = []

    for chunk, id in zip(chunks, ids):     
        if id not in unique_ids:       
            unique_ids.add(id)
            unique_chunks.append(chunk) 

    print(f"🧱 Creating Chroma vectorstore with {len(unique_chunks)} unique chunks...")

    vectorstore = Chroma.from_documents(
        documents=unique_chunks,
        ids=list(unique_ids),
        embedding=embedding_function,
        persist_directory=vectorstore_path
    )

    return vectorstore

# Build vectorstore
vectorstore = create_vectorstore(
    chunks=chunks,
    embedding_function=embedding_function,
    vectorstore_path="vectorstore_test"
)

✅ Embedding vector length: 1536
🔍 Distance (machine learning vs AI): 0.4123
🔍 Distance (data mining vs AI): 0.6109
🧱 Creating Chroma vectorstore with 9 unique chunks...


## 🔎 Step 2: Query the Vectorstore for Relevant Chunks

Here we:
- Reload our Chroma vector database from disk
- Create a retriever to search for the most relevant chunks based on a user query
- Perform a sample query to extract the article title and preview the matching content

In [6]:
## 🔎 2. Query for Relevant Data

from langchain_community.vectorstores import Chroma  # ✅ Consistent import

# Load the existing vectorstore
vectorstore = Chroma(
    persist_directory="vectorstore_test",  # Match your previous save location
    embedding_function=embedding_function
)

# Create a retriever from the vectorstore
retriever = vectorstore.as_retriever(search_type="similarity")

# Query for the most relevant chunks
query = "What is the title of the article?"
relevant_chunks = retriever.invoke(query)

# Print a summary of the results
print(f"🔍 Retrieved {len(relevant_chunks)} relevant chunks for query: '{query}'")
print(relevant_chunks[0].page_content[:500])  # Preview first result

C:\Users\sulta\AppData\Local\Temp\ipykernel_63120\1640775561.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


🔍 Retrieved 4 relevant chunks for query: 'What is the title of the article?'
was unnecessary and thus surprising readers with the relative disﬂuency of the text.
Both the experts and prevailing wisdom present plausible views, but which (if either) is
correct? The present paper provides an empirical investigation of the strategy of complex-
ity, and ﬁnds such a strategy to be unsuccessful. Five studies demonstrate that the loss of
ﬂuency due to needless complexity in a text negatively impacts raters’ assessments of the
text’s authors.
EXPERIMENT 1
Experiment 1 aimed to an


## 🧠 Step 3: Generate a Prompt for the Language Model

In this step, we:
- Define a structured prompt template to guide the model’s response
- Concatenate retrieved context chunks to form a clean input
- Fill in the template with both context and query to create the final prompt for the LLM

In [7]:
## 🧠 3. Generate Response from Retrieved Context

from langchain_core.prompts import ChatPromptTemplate

# Define the prompt template
PROMPT_TEMPLATE = """
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer
the question. If you don't know the answer, say that you
don't know. DON'T MAKE UP ANYTHING.

{context}

---

Answer the question based on the above context: {question}
"""

# Safely concatenate context text from relevant chunks
if relevant_chunks:
    context_text = "\n\n---\n\n".join([doc.page_content for doc in relevant_chunks])
else:
    context_text = "No context found."

# Create and format the prompt
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(
    context=context_text,
    question="What is the title of the paper?"
)

# Print formatted prompt
print("📋 Prompt Preview:\n")
print(prompt)

📋 Prompt Preview:

Human: 
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer
the question. If you don't know the answer, say that you
don't know. DON'T MAKE UP ANYTHING.

was unnecessary and thus surprising readers with the relative disﬂuency of the text.
Both the experts and prevailing wisdom present plausible views, but which (if either) is
correct? The present paper provides an empirical investigation of the strategy of complex-
ity, and ﬁnds such a strategy to be unsuccessful. Five studies demonstrate that the loss of
ﬂuency due to needless complexity in a text negatively impacts raters’ assessments of the
text’s authors.
EXPERIMENT 1
Experiment 1 aimed to answer several simple questions. First, does increasing the
complexity of text succeed in making the author appear more intelligent? Second, to
what extent does the success of this strategy depend on the quality of the original, simpler
writing? Finally, if the strategy is 

## 🧠 Step 4: Generate the Final Answer

Now that we have the prompt ready, we use the LLM to generate the response based on the provided context. This gives us the structured answer to our question (e.g., the title of the paper).

In [8]:
## 🚀 Invoke the LLM to Generate the Answer

# Generate the answer using the language model
response = llm.invoke(prompt)

# Display the response
print("🧠 LLM Response:\n")
print(response)

🧠 LLM Response:

content='The title of the paper is "Consequences of Erudite Vernacular Utilized Irrespective of Necessity: Problems with Using Long Words Needlessly."' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 1337, 'total_tokens': 1369, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-C6oufqqWrxXVPtKEGMcJrgnZGruoN', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--33bb4f2d-2c7f-485c-b898-4a2d0eeb1eb0-0' usage_metadata={'input_tokens': 1337, 'output_tokens': 32, 'total_tokens': 1369, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## 🧠 Advanced RAG Pipeline with Structured Output

In this block, we:
- Use LangChain’s Expression Language to compose a RAG pipeline in a clean, modular way
- Define `Pydantic` models to enforce a structured response format (e.g., title, summary, authors)
- Use `with_structured_output` to generate validated and organized outputs from the LLM

In [14]:
### ⚙️ LangChain Expression Language for RAG

from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field

# Function to format retrieved documents into context string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Improved function to get comprehensive context for structured extraction
def get_comprehensive_context(question):
    """Get comprehensive context by searching for key paper elements"""
    # Search for different aspects to ensure we get all relevant chunks
    searches = [
        "title consequences erudite vernacular",
        "authors Daniel Oppenheimer",
        "abstract summary introduction",
        "publication year 2005 copyright",
        "conclusion implications applications"
    ]
    
    all_chunks = []
    seen_content = set()
    
    for search_query in searches:
        chunks = vectorstore.similarity_search(search_query, k=3)
        for chunk in chunks:
            # Avoid duplicate chunks
            chunk_preview = chunk.page_content[:100]
            if chunk_preview not in seen_content:
                seen_content.add(chunk_preview)
                all_chunks.append(chunk)
    
    # Also include the best chunks from the original question
    question_chunks = vectorstore.similarity_search(question, k=2)
    for chunk in question_chunks:
        chunk_preview = chunk.page_content[:100]
        if chunk_preview not in seen_content:
            seen_content.add(chunk_preview)
            all_chunks.append(chunk)
    
    return format_docs(all_chunks)

# Step 1: Basic RAG Chain (No structure)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
)

# Run basic RAG query
basic_response = rag_chain.invoke("What's the title of this paper?")
print("🧠 Basic Answer:\n", basic_response.content)

### 🧱 Define Structured Output Models

class AnswerWithSources(BaseModel):
    """An answer to the question, with sources and reasoning."""
    answer: str = Field(description="Answer to question")
    sources: str = Field(description="Full direct text chunk from the context used to answer the question")
    reasoning: str = Field(description="Explain the reasoning of the answer based on the sources")

class ExtractedInfo(BaseModel):
    """Extracted information about the research article"""
    paper_title: AnswerWithSources
    paper_summary: AnswerWithSources
    publication_year: AnswerWithSources
    paper_authors: AnswerWithSources

# Create a specialized prompt for structured extraction
STRUCTURED_PROMPT_TEMPLATE = """
You are an expert research paper analyzer. Extract the following information from the provided context.
Be precise and only use information that is explicitly stated in the context.

Context:
{context}

Extract the following information about this research paper:
1. Paper Title: The exact title of the research paper
2. Paper Summary: A comprehensive summary of the paper's main findings and contributions  
3. Publication Year: The year this paper was published
4. Paper Authors: The authors who wrote this paper

For each field, provide:
- answer: The extracted information
- sources: Quote the exact text from context that supports your answer
- reasoning: Brief explanation of how you found this information

If any information is not available in the context, state "Not available in provided context" for that field.

Question: {question}
"""

# Create new prompt template for structured extraction
structured_prompt_template = ChatPromptTemplate.from_template(STRUCTURED_PROMPT_TEMPLATE)

# Step 2: Improved RAG Chain with comprehensive context retrieval
structured_rag_chain = (
    {"context": get_comprehensive_context, "question": RunnablePassthrough()}
    | structured_prompt_template
    | llm.with_structured_output(ExtractedInfo, strict=True)
)

# Run structured query
structured_response = structured_rag_chain.invoke(
    "Extract the title, summary, publication year, and authors from this research paper."
)

# Display structured output
from pprint import pprint
print("\n📦 Structured Output:")
pprint(structured_response.model_dump())

# Quick verification - show what context was actually used
print("\n🔍 Context Used for Structured Extraction:")
context_used = get_comprehensive_context("Extract paper information")
print(f"Total context length: {len(context_used)} characters")
print("First 500 characters:")
print(context_used[:500])

🧠 Basic Answer:
 The title of the paper is "Consequences of Erudite Vernacular Utilized Irrespective of Necessity: Problems with Using Long Words Needlessly."

📦 Structured Output:
{'paper_authors': {'answer': 'DANIEL M. OPPENHEIMER',
                   'reasoning': 'The authorship is clearly listed in the text, '
                                "along with the author's affiliation.",
                   'sources': 'DANIEL M. OPPENHEIMER* Princeton University, '
                              'USA'},
 'paper_summary': {'answer': 'The paper explores the effectiveness of using '
                             'complex vocabulary among undergraduate writers, '
                             'finding that increased complexity does not '
                             'enhance perceived intelligence. Experiments '
                             'revealed a negative relationship between text '
                             'complexity and judged intelligence, regardless '
                             "

## 📊 Convert Structured LLM Output to a Data Table

Here we:
- Re-invoke the structured RAG pipeline to get a nested dictionary response
- Flatten the response to extract `answer`, `source`, and `reasoning` fields
- Format it into a readable table with `pandas`, showing each field as a row across the extracted attributes

---

In [16]:
### 📊 Transform Structured Response into a DataFrame

# Use the existing structured_response (no need to re-invoke)
# structured_response = structured_rag_chain.invoke(...)  # Remove this line

# Create a cleaner approach to build the DataFrame
data = {
    'Title': [
        structured_response.paper_title.answer,
        structured_response.paper_title.sources,
        structured_response.paper_title.reasoning
    ],
    'Summary': [
        structured_response.paper_summary.answer,
        structured_response.paper_summary.sources,
        structured_response.paper_summary.reasoning
    ],
    'Year': [
        structured_response.publication_year.answer,
        structured_response.publication_year.sources,
        structured_response.publication_year.reasoning
    ],
    'Authors': [
        structured_response.paper_authors.answer,
        structured_response.paper_authors.sources,
        structured_response.paper_authors.reasoning
    ]
}

# Create DataFrame with clear row labels
structured_response_df = pd.DataFrame(
    data,
    index=['Answer', 'Source', 'Reasoning']
)

# Display the table
print("📊 Extracted Information Table:")
structured_response_df

📊 Extracted Information Table:


,Title,Summary,Year,Authors
Answer,Consequences of Erudite Vernacular Utilized Ir...,The paper examines the strategy of using compl...,2006,Daniel M. Oppenheimer
Source,Consequences of Erudite Vernacular Utilized Ir...,The present paper provides an empirical invest...,Appl. Cognit. Psychol. 20: 139–156 (2006),"DANIEL M. OPPENHEIMER* Princeton University, USA"
Reasoning,The title is explicitly stated at the beginnin...,The summary distills the key findings and cont...,The publication year is given in the citation ...,The authorship is clearly stated along with th...
